# 🚀 Inference with AWQ Quantized Models

This notebook demonstrates **three methods** to run inference with an AWQ-quantized Qwen3-4B-Instruct model:

1. **AutoAWQ Native** — direct from AutoAWQ library
2. **HuggingFace Transformers** — using the standard HF API
3. **vLLM** — high-throughput production serving

**Prerequisites:** An AWQ-quantized model (from the [quantization notebook](quantize_qwen3_awq.ipynb))

In [ ]:
# Configuration
MODEL_PATH = "./qwen3-4b-instruct-awq"  # Path to your quantized model
# Or use a pre-quantized model from HuggingFace:
# MODEL_PATH = "Qwen/Qwen3-4B-AWQ"

---
## Method 1: AutoAWQ Native

The most direct approach using AutoAWQ's optimized kernels with layer fusion.

In [ ]:
import torch
import time
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

print("Loading model with AutoAWQ...")
start = time.time()

awq_model = AutoAWQForCausalLM.from_quantized(
    MODEL_PATH,
    fuse_layers=True,    # Fuse QKV + MLP for faster inference
    device_map="auto"
)
awq_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

load_time = time.time() - start
print(f"Loaded in {load_time:.1f}s")

# Check memory
for i in range(torch.cuda.device_count()):
    mem = torch.cuda.memory_allocated(i) / 1e9
    print(f"GPU {i}: {mem:.2f} GB allocated")

In [ ]:
def generate_awq(model, tokenizer, prompt, system=None, max_tokens=256, temperature=0.7):
    """Generate text using AutoAWQ model."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_len = inputs.input_ids.shape[1]
    
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=temperature > 0
        )
    gen_time = time.time() - start
    
    output_len = outputs.shape[1] - input_len
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    
    return response, {
        "gen_time": gen_time,
        "output_tokens": output_len,
        "tokens_per_sec": output_len / gen_time
    }

In [ ]:
# Test AutoAWQ inference
response, stats = generate_awq(
    awq_model, awq_tokenizer,
    prompt="Explain the difference between machine learning and deep learning.",
    system="You are a helpful AI teacher. Be concise."
)

print("Response:")
print(response)
print(f"\n[{stats['output_tokens']} tokens | {stats['gen_time']:.1f}s | {stats['tokens_per_sec']:.1f} tok/s]")

In [ ]:
# Clean up to free memory for next method
del awq_model
torch.cuda.empty_cache()
print("AutoAWQ model unloaded.")

---
## Method 2: HuggingFace Transformers

Since Transformers v4.35+, AWQ models are natively supported. No extra configuration needed.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading model with HuggingFace Transformers...")
start = time.time()

hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto"
)
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

load_time = time.time() - start
print(f"Loaded in {load_time:.1f}s")

# Show quantization config detected by Transformers
if hasattr(hf_model.config, 'quantization_config'):
    print(f"Quantization config: {hf_model.config.quantization_config}")

In [ ]:
# Test HuggingFace inference
messages = [
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "Write a Python function that checks if a number is prime."}
]

text = hf_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = hf_tokenizer(text, return_tensors="pt").to(hf_model.device)
input_len = inputs.input_ids.shape[1]

start = time.time()
with torch.no_grad():
    outputs = hf_model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True
    )
gen_time = time.time() - start
output_len = outputs.shape[1] - input_len

response = hf_tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
print("Response:")
print(response)
print(f"\n[{output_len} tokens | {gen_time:.1f}s | {output_len/gen_time:.1f} tok/s]")

In [ ]:
# Using Transformers pipeline API (even simpler)
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=hf_model,
    tokenizer=hf_tokenizer
)

messages = [{"role": "user", "content": "What are the three laws of robotics?"}]
output = pipe(messages, max_new_tokens=200, temperature=0.7, do_sample=True)
print(output[0]["generated_text"][-1]["content"])

In [ ]:
# Clean up
del hf_model, pipe
torch.cuda.empty_cache()
print("Transformers model unloaded.")

---
## Method 3: vLLM

vLLM provides the highest throughput with PagedAttention and continuous batching.

First, install vLLM: `pip install vllm>=0.6.1`

In [ ]:
# Option A: Python API
try:
    from vllm import LLM, SamplingParams
    
    print("Loading model with vLLM...")
    start = time.time()
    
    llm = LLM(
        model=MODEL_PATH,
        quantization="awq",
        dtype="float16",
        gpu_memory_utilization=0.8,
        max_model_len=4096
    )
    
    load_time = time.time() - start
    print(f"Loaded in {load_time:.1f}s")
    
    # Generate
    sampling_params = SamplingParams(
        temperature=0.7,
        top_p=0.9,
        max_tokens=256
    )
    
    prompts = [
        "Explain quantum computing in simple terms.",
        "Write a haiku about artificial intelligence.",
        "What is the meaning of life?"
    ]
    
    start = time.time()
    outputs = llm.generate(prompts, sampling_params)
    gen_time = time.time() - start
    
    total_tokens = 0
    for output in outputs:
        response = output.outputs[0].text
        tokens = len(output.outputs[0].token_ids)
        total_tokens += tokens
        print(f"\nPrompt: {output.prompt[:60]}...")
        print(f"Response: {response[:200]}{'...' if len(response) > 200 else ''}")
        print(f"[{tokens} tokens]")
    
    print(f"\nTotal: {total_tokens} tokens in {gen_time:.1f}s = {total_tokens/gen_time:.1f} tok/s")
    
    del llm
    torch.cuda.empty_cache()
    
except ImportError:
    print("vLLM not installed. Install with: pip install vllm>=0.6.1")
    print("Skipping vLLM example.")

### Option B: vLLM as OpenAI-Compatible Server

Start the server in a terminal:
```bash
vllm serve ./qwen3-4b-instruct-awq \
    --quantization awq \
    --dtype float16 \
    --host 0.0.0.0 \
    --port 8000
```

Then query with the OpenAI client:

In [ ]:
# Uncomment to use after starting vLLM server
# from openai import OpenAI
#
# client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
#
# response = client.chat.completions.create(
#     model=MODEL_PATH,
#     messages=[
#         {"role": "system", "content": "You are a helpful assistant."},
#         {"role": "user", "content": "Explain AWQ quantization."}
#     ],
#     max_tokens=256,
#     temperature=0.7
# )
# print(response.choices[0].message.content)

---
## Performance Summary

| Method | Load Time | Throughput (batch=1) | Best For |
|--------|-----------|---------------------|----------|
| AutoAWQ | Fast | High | Development, testing |
| Transformers | Fast | Medium | HF ecosystem integration |
| vLLM | Moderate | Highest | Production serving |

## ✅ Done!

You've learned how to run inference with AWQ-quantized models using three different methods.

**Recommendations:**
- **Development/Testing** → AutoAWQ or Transformers
- **Production API Server** → vLLM
- **HuggingFace Ecosystem** → Transformers